# Milestone 2: LoRA fine-tuning, checkpoint selection, before/after, final test
Kaggle: **GPU T4 x2, Internet ON**. Total ~45-60 min.

In [ ]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'   # one GPU: simpler, reproducible (no DataParallel)
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
!rm -rf trial-eligibility-transformer
!git clone https://github.com/UsmanBari/trial-eligibility-transformer.git
%cd trial-eligibility-transformer
!pip install -q -r requirements.txt && pip install -q -e .
!python -m pytest -q tests

In [ ]:
# Rebuilds the identical split (deterministic) and checks it against the manifest in the repo
!python scripts/prepare_data.py
!git diff --stat data/splits/split_manifest.json && echo 'manifest unchanged = split reproduced exactly'

## 1. Zero-shot baseline on validation (the 'before' model)

In [ ]:
!python scripts/evaluate.py --model Qwen/Qwen2.5-1.5B-Instruct --split val --tag baseline_qwen25_1p5b

## 2. LoRA fine-tuning (train split only; val loss each epoch)

In [ ]:
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'  # less memory fragmentation
!python scripts/train_lora.py

## 3. Evaluate every epoch checkpoint on validation, pick the best (by val macro-F1)

In [ ]:
import glob, subprocess
ckpts = sorted(glob.glob('outputs/lora_qwen25_1p5b/checkpoint-*'), key=lambda p: int(p.split('-')[-1]))
assert ckpts, 'no checkpoints found: the training cell above failed, fix that first'
tags = []
for i, ck in enumerate(ckpts, 1):
    tag = f'ft_ep{i}_val'; tags.append(tag)
    subprocess.run(['python', 'scripts/evaluate.py', '--adapter', ck, '--split', 'val', '--tag', tag], check=True)
subprocess.run(['python', 'scripts/select_checkpoint.py', '--run', 'outputs/lora_qwen25_1p5b', '--tags', *tags], check=True)

## 4. FINAL held-out TEST (run once, only after steps 1-3 are done)
Same model, same greedy decoding, before vs after fine-tuning.

In [ ]:
!python scripts/evaluate.py --model Qwen/Qwen2.5-1.5B-Instruct --split test --final-test --tag baseline_qwen25_1p5b_test
!python scripts/evaluate.py --adapter outputs/lora_qwen25_1p5b_best --split test --final-test --tag ft_best_test

## 5. Error analysis + figures

In [ ]:
!python scripts/error_analysis.py results/ft_best_test
best = __import__('json').load(open('results/checkpoint_selection.json'))['selected']['tag']
!python scripts/error_analysis.py results/{best}
!python scripts/make_figures.py --train-log outputs/lora_qwen25_1p5b/train_log.json --cm baseline_qwen25_1p5b_test ft_best_test --compare baseline_qwen25_1p5b_test ft_best_test --compare-names 'Zero-shot' 'Fine-tuned (LoRA)'

In [ ]:
import json
for t in ['baseline_qwen25_1p5b', best, 'baseline_qwen25_1p5b_test', 'ft_best_test']:
    m = json.load(open(f'results/{t}/metrics.json'))
    o, i, e = m['overall'], m['inclusion'], m['exclusion']
    print(f"{t:28s} acc={o['classification']['accuracy']:.3f} mF1={o['classification']['macro_f1']:.3f} "
          f"incl={i['classification']['accuracy']:.3f} excl={e['classification']['accuracy']:.3f} "
          f"invalid={o['classification']['n_invalid']} evF1={o['evidence']['micro']['f1']:.3f}")
print(open('outputs/lora_qwen25_1p5b/train_config.json').read())

## 6. Download
`m2_results.zip` (results + figures + logs, small) and `lora_best_adapter.zip` (~70 MB adapter) from the Output panel.

In [ ]:
!cp outputs/lora_qwen25_1p5b/train_log.json outputs/lora_qwen25_1p5b/train_config.json results/
!zip -qr /kaggle/working/m2_results.zip results data/splits
!cd outputs && zip -qr /kaggle/working/lora_best_adapter.zip lora_qwen25_1p5b_best
!ls -la /kaggle/working